# Energy Algorithms - Optimization Portfolio Walkthrough

**Target Audience:** Euphemia   Junior Optimization Engineer / Industry algorithmic trading

**Modules:** Energy Markets (PCR/Euphemia), LP/MIP Optimization, Backtesting, ENTSO-E Data

## Setup - Imports

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if "notebooks" in os.getcwd() else ".")
import numpy as np
from energy_markets.pcr_model import PCRModel
from energy_markets.multi_zone import demo_multi_zone
from energy_markets.market_clearing import demo_clearing
from energy_markets.intraday import demo_intraday
from lp_optimization.transportation import demo_transportation
from lp_optimization.portfolio import demo_portfolio
from lp_optimization.scheduling import demo_uc
from lp_optimization.storage import demo_storage
from backtester.engine import backtest
from strategies.momentum import momentum
from strategies.mean_reversion import mean_reversion
from strategies.sma_crossover import sma_crossover
from energy_data.fetcher import fetch_demo_day_ahead, fetch_demo_generation_mix
print('All modules imported')

## 1. Energy Markets - PCR & Euphemia Connection

### 1.1 Market Clearing

In [ ]:
r = demo_clearing()
print(f"Price: EUR {r["clearing_price"]:.0f}/MWh | Volume: {r["clearing_volume"]:.0f} MWh")
print(f"Social Welfare: EUR {r["social_welfare"]:,.0f}")
for s in r['accepted_supply']:
    print(f"  {s["id"]:10s} EUR {s["price"]:>6.0f}/MWh x {s["quantity"]:>5.0f} MWh")

### 1.2 PCR Model - Block Orders

In [ ]:
m = PCRModel(area="IT")
m.add_supply('wind', 5, 300); m.add_supply('gas', 80, 200); m.add_supply('diesel', 120, 100)
m.add_demand('base', 200, 350); m.add_demand('peak', 150, 150)
m.add_block("hydro_A", 35, 100, group="hydro"); m.add_block("hydro_B", 40, 50, group="hydro")
m.add_block("cfg1", 60, 80, group="excl_A"); m.add_block("cfg2", 55, 80, group="excl_A")
r = m.solve(); m.report()
print(f"MCP: EUR {r["mcp"]:.0f}/MWh | Welfare: EUR {r["welfare"]:,.0f}")

### 1.3 Multi-Zone Coupling - ATC Flows

In [ ]:
r = demo_multi_zone()
print(f"Welfare: EUR {r["welfare"]:,.0f}")
for f, mw in r['flows'].items(): print(f'  {f}: {mw} MW')
for zn, zd in r['zones'].items(): print(f'  {zn}: MCP=EUR {zd["mcp"]}/MWh')

### 1.4 Intraday Continuous Trading

In [ ]:
r = demo_intraday()
print(f"{len(r["trades"])} trades | {r["total_volume"]} MW | VWAP: EUR {r["vwap"]:.2f}")
for t in r['trades'][:5]: print(f'  t={t["time"]:5.1f}h EUR {t["price"]:7.1f} x {t["qty"]:5.0f} MW')

## 2. LP/MIP Optimization

### 2.1 Transportation

In [ ]:
r = demo_transportation()
print(f"Cost: EUR {r["total_cost"]:,.0f}")
for (w,rt), q in r['allocations'].items(): print(f'  {w} -> {rt}: {q:.0f}')

### 2.2 Portfolio Optimization - Mean-Variance

In [ ]:
r = demo_portfolio()
print(f"Return: {r["return"]:.2%} | Risk: {r["risk"]:.2%} | Sharpe: {r["return"]/r["risk"]:.2f}")
for i,w in enumerate(r['weights']):
    if w > 0.001: print(f'  Asset {i+1}: {w:>6.1%}')

### 2.3 Unit Commitment - MIP

In [ ]:
r = demo_uc()
print(f"Cost: EUR {r["total_cost"]:,.0f}")
for tk, p in r['schedule'].items():
    t = int(tk.split("=")[1]); on = p["_online"]
    gen = sum(v for k,v in p.items() if not k.startswith("_"))
    print(f"  t={t:>2}: demand={p["_demand"]:5.0f} gen={gen:5.0f} online={on}")

### 2.4 BESS Storage Optimization

In [ ]:
r = demo_storage()
print(f"Revenue: EUR {r["revenue"]:,.2f} | Cycles: {r.get("total_cycles","N/A")}")
for p in r['schedule'][:12]:
    print(f"  H{p["hour"]:>2}: EUR {p["price"]:6.1f} chg={p["charge"]:6.1f} dis={p["discharge"]:6.1f} SoC={p["soc"]:6.1f}")
print("  ... (12 more hours)")

## 3. Backtesting Engine - Vectorized, No Look-Ahead Bias

In [ ]:
np.random.seed(42); n = 252*2
prices = 100 * np.cumprod(1 + np.random.normal(0.0002, 0.015, n))
for name, sig in [('Momentum',momentum(prices)),('MeanRev',mean_reversion(prices)),('SMA',sma_crossover(prices))]:
    r = backtest(prices, sig)
    print(f'  {name:<12} Ret={r["total_return"]:>7.1%} Sharpe={r["sharpe"]:>6.2f} MaxDD={r["max_drawdown"]:>7.1%} Trades={r["n_trades"]}')

## 4. ENTSO-E Data Pipeline

In [ ]:
p = fetch_demo_day_ahead()
print(f"{p["area"]} {p["date"]}: Avg EUR {p["avg_price"]}/MWh | Range EUR {p["min_price"]}-{p["max_price"]}")
g = fetch_demo_generation_mix()
print(f"Generation mix ({g["total_mw"]} MW):")
for x in g['generation']: print(f'  {x["type"]:<25} {x["mw"]:>6.0f}MW ({x["mw"]/g["total_mw"]*100:>5.1f}%)')

## Summary

| Skill | Module |
|-------|--------|
| Social welfare LP | `energy_markets/pcr_model.py` |
| Block orders | `energy_markets/block_orders.py` |
| Multi-zone coupling | `energy_markets/multi_zone.py` |
| Intraday simulation | `energy_markets/intraday.py` |
| Unit commitment MIP | `lp_optimization/scheduling.py` |
| BESS storage LP | `lp_optimization/storage.py` |
| Portfolio optimization | `lp_optimization/portfolio.py` |
| Vectorized backtesting | `backtester/engine.py` |
| ENTSO-E pipeline | `energy_data/fetcher.py` |
| 40 tests | `tests/` |

**Run all tests:** `pytest tests/ -v` (40 tests passing)

**For Euphemia   interviewers:** See `energy_markets/EUPHEMIA_INTERVIEW.md`